In [1]:
import pickle
import regex as re

In [2]:
from tests.test_tokenizer import get_tokenizer_from_vocab_merges_path

In [3]:
VOCAB_PATH = "../tests/fixtures/gpt2_vocab.json"
MERGES_PATH = "../tests/fixtures/gpt2_merges.txt"


In [4]:
# merges = pickle.load(open("../TinyStoriesV2-GPT4-train.txt.merges.pkl", 'rb'))
# vocab = pickle.load(open("../TinyStoriesV2-GPT4-train.txt.vocab.pkl", 'rb'))
merges = get_tokenizer_from_vocab_merges_path(VOCAB_PATH, MERGES_PATH).merges
vocab = get_tokenizer_from_vocab_merges_path(VOCAB_PATH, MERGES_PATH).vocab

In [5]:
vocab[9999]

b'ordon'

In [6]:
len(merges)

50000

In [7]:
len(vocab)

50257

In [8]:
max(vocab, key=lambda o: len(vocab[o]))

35496

In [9]:
vocab[7162]

b' Conserv'

`merges` -> list of tuples -> tuples having byte(s) pairs

sentence -> pre-tokenize -> words

words -> break into bytes -> merge each pair in the order of their occurrence in `merges`

In [10]:
x = " this word is bussing"

In [11]:
xp = tuple([bytes([o]) for o in list(" this".encode("utf-8"))])
xp

(b' ', b't', b'h', b'i', b's')

In [13]:
merges[:50]

[(b' ', b't'),
 (b' ', b'a'),
 (b'h', b'e'),
 (b'i', b'n'),
 (b'r', b'e'),
 (b'o', b'n'),
 (b' t', b'he'),
 (b'e', b'r'),
 (b' ', b's'),
 (b'a', b't'),
 (b' ', b'w'),
 (b' ', b'o'),
 (b'e', b'n'),
 (b' ', b'c'),
 (b'i', b't'),
 (b'i', b's'),
 (b'a', b'n'),
 (b'o', b'r'),
 (b'e', b's'),
 (b' ', b'b'),
 (b'e', b'd'),
 (b' ', b'f'),
 (b'in', b'g'),
 (b' ', b'p'),
 (b'o', b'u'),
 (b' a', b'n'),
 (b'a', b'l'),
 (b'a', b'r'),
 (b' t', b'o'),
 (b' ', b'm'),
 (b' o', b'f'),
 (b' ', b'in'),
 (b' ', b'd'),
 (b' ', b'h'),
 (b' an', b'd'),
 (b'i', b'c'),
 (b'a', b's'),
 (b'l', b'e'),
 (b' t', b'h'),
 (b'i', b'on'),
 (b'o', b'm'),
 (b'l', b'l'),
 (b'en', b't'),
 (b' ', b'n'),
 (b' ', b'l'),
 (b's', b't'),
 (b' ', b're'),
 (b'v', b'e'),
 (b' ', b'e'),
 (b'r', b'o')]

In [14]:
xp = tuple([bytes([o]) for o in list("intelligence".encode("utf-8"))])
print("initially: ", xp)

i = 0
while len(xp) > 1 and i+1 < len(xp):
    # print(f"checking: {(xp[i], xp[i+1])}; i: {i}; xp[{i}]: {xp[i]}")
    print("checking: ", xp[i], xp[i+1])
    for p in merges:
        if (xp[i], xp[i+1]) == p:
            # print("found: ", i)
            xp = (*xp[:i], xp[i]+xp[i+1], *xp[i+2:])
            i = 0 # start searching from the beginning again
            print("merged: ", xp)
            break
    else: # if pair is not found in merges
        i += 1

xp

initially:  (b'i', b'n', b't', b'e', b'l', b'l', b'i', b'g', b'e', b'n', b'c', b'e')
checking:  b'i' b'n'
merged:  (b'in', b't', b'e', b'l', b'l', b'i', b'g', b'e', b'n', b'c', b'e')
checking:  b'in' b't'
merged:  (b'int', b'e', b'l', b'l', b'i', b'g', b'e', b'n', b'c', b'e')
checking:  b'int' b'e'
checking:  b'e' b'l'
merged:  (b'int', b'el', b'l', b'i', b'g', b'e', b'n', b'c', b'e')
checking:  b'int' b'el'
merged:  (b'intel', b'l', b'i', b'g', b'e', b'n', b'c', b'e')
checking:  b'intel' b'l'
checking:  b'l' b'i'
merged:  (b'intel', b'li', b'g', b'e', b'n', b'c', b'e')
checking:  b'intel' b'li'
checking:  b'li' b'g'
checking:  b'g' b'e'
merged:  (b'intel', b'li', b'ge', b'n', b'c', b'e')
checking:  b'intel' b'li'
checking:  b'li' b'ge'
checking:  b'ge' b'n'
checking:  b'n' b'c'
merged:  (b'intel', b'li', b'ge', b'nc', b'e')
checking:  b'intel' b'li'
checking:  b'li' b'ge'
checking:  b'ge' b'nc'
checking:  b'nc' b'e'


(b'intel', b'li', b'ge', b'nc', b'e')

In [19]:
xp = tuple([bytes([o]) for o in list("s".encode("utf-8"))])
print("initially: ", xp)

i = 0
while i < len(merges) :
    j = 0
    while j+1 < len(xp):
        print("checking: ", xp[j], xp[j+1])
        if merges[i] == (xp[j], xp[j+1]):
            xp = (*xp[:j], xp[j]+xp[j+1], *xp[j+2:])
            i = 0
            print("merged: ", xp)
            break
        j += 1
    else:
        i += 1

initially:  (b's',)


In [5]:
rvocab = {v:k for k,v in vocab.items()}

In [184]:
rvocab[b'ho']

6940

In [14]:
def encode_subword(subword: str):
    sb = tuple([bytes([o]) for o in list(subword.encode("utf-8"))])

    i = 0
    while i < len(merges):
        j = 0
        if len(sb) == 1:
            return [rvocab[sb[0]]]
        while j + 1 < len(sb):
            if merges[i] == (sb[j], sb[j + 1]):
                sb = (*sb[:j], sb[j] + sb[j + 1], *sb[j + 2:])
                i = 0
                break
            j += 1
        else:
            i += 1
    return [rvocab[o] for o in sb]

In [176]:
encode_subword("honestly")

[6940, 6832, 350, 460]

In [18]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [19]:
x = "Hello, how <|endoftext|><|endoftext|> are you?<|endoftext|>🙃"

In [20]:
subwords = []
for subword in re.finditer(PAT, x, re.IGNORECASE):
    subwords.append(subword.captures()[0])

subwords

['Hello',
 ',',
 ' how',
 ' <|',
 'endoftext',
 '|><|',
 'endoftext',
 '|>',
 ' are',
 ' you',
 '?<|',
 'endoftext',
 '|>🙃']

In [196]:
for o in subwords:
    print(encode_subword(o))

[726]
[1034, 114, 100]
[430]
[861, 559, 298]


In [44]:
special_tokens = ["<|endoftext|>", "<|endoftext|><|endoftext|>"]
special_tokens.sort(key=lambda o: len(o), reverse=True)
special_tokens

['<|endoftext|><|endoftext|>', '<|endoftext|>']

In [45]:
escaped_tokens = [re.escape(o) for o in special_tokens]
special_tok_pat = '(' + '|'.join(escaped_tokens) + ')'
raw_parts = re.split(special_tok_pat, x)
raw_parts

['Hello, how ',
 '<|endoftext|><|endoftext|>',
 ' are you?',
 '<|endoftext|>',
 '🙃']

In [35]:
tagged_parts = []
for part in raw_parts:
    if part in special_tokens:
        tagged_parts.append((None, part))
    else:
        tagged_parts.append((part, None))

tagged_parts

[('Hello, how ', None),
 (None, '<|endoftext|><|endoftext|>'),
 (' are you?', None),
 (None, '<|endoftext|>'),
 ('🙃', None)]

In [ ]:
def encode(text: str):
    subwords = []
    for subword in re.finditer(PAT, text, re.IGNORECASE):
        subwords.append(subword.captures()[0])

    token_ids = []
    for subword in subwords:
        token_ids.extend(encode_subword(subword))
    
    return token_ids

encode("Hello, how <|endoftext|><|endoftext|> are you?<|endoftext|>")

[726, 1034, 114, 100, 430, 861, 559, 298]

In [204]:
t = encode(" this shit's bussin'")
t

[726, 1034, 114, 100, 430, 861, 559, 298]

In [208]:
for o in t:
    print(vocab[o])

b' this'
b' wo'
b'r'
b'd'
b' is'
b' bu'
b'ss'
b'ing'


In [3]:
text1 = "hi this is a test<|endoftext|>this is another sentence"
text2 = "hi this is a test. there is not another sentence ... wait"

In [17]:
escaped_tokens = [re.escape(o) for o in ["<|endoftext|>"]]
special_tok_pat = '|'.join(escaped_tokens)
special_tok_pat

'<\\|endoftext\\|>'

In [22]:
re.split(special_tok_pat, text1)

['hi this is a test', 'this is another sentence']

In [28]:
PAT + '|' + special_tok_pat

"'(?:[sdmt]|ll|ve|re)| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+|<\\|endoftext\\|>"

In [31]:
subwords = []
for part in re.split(special_tok_pat, text1):
    for subword in re.finditer(PAT, part, re.IGNORECASE):
        subwords.append(subword.captures()[0])

In [30]:
subwords

['hi', ' this', ' is', ' a', ' test', 'this', ' is', ' another', ' sentence']

In [36]:
for part in re.finditer(special_tok_pat, text1):
    print(part)

<regex.Match object; span=(17, 30), match='<|endoftext|>'>


In [40]:
text1[30]

't'

In [6]:
from tokenizer import Tokenizer

In [8]:
tok = Tokenizer(
    vocab=vocab,
    merges=merges,
    special_tokens=["<|endoftext|>", "<|endoftext|><|endoftext|>"]
)

In [9]:
tok.rvocab[b"<|endoftext|>"], tok.rvocab[b"<|endoftext|><|endoftext|>"]


(50259, 50258)

In [9]:
test_string = "Hello, how <|endoftext|><|endoftext|> are you?<|endoftext|>🙃"

In [10]:
ids = tok.encode(test_string)
ids

[15496, 11, 703, 220, 50256, 50256, 389, 345, 30, 50256, 8582, 247, 225]

In [11]:
[tok.decode([o]) for o in ids]

['Hello',
 ',',
 ' how',
 ' ',
 '<|endoftext|>',
 '<|endoftext|>',
 ' are',
 ' you',
 '?',
 '<|endoftext|>',
 '�',
 '�',
 '�']

In [25]:
bytes.decode(vocab[ids[3]]+vocab[ids[4]]+vocab[ids[5]], errors="replace")

'ò h'

In [43]:
bytes.decode(b''.join([vocab[o] for o in ids]), errors="replace")

'Héllò hôw are ü? 🙃'

In [21]:
"hi".encode("utf-8")

b'hi'

In [10]:
"🙃".encode()

b'\xf0\x9f\x99\x83'

In [5]:
text1 = "hi this is a test<|endoftext|>this is another sentence"

In [6]:
text1

'hi this is a test<|endoftext|>this is another sentence'

In [7]:
out = ""
for id in tok.encode(text1):
    out += bytes.decode(vocab[id], errors='replace')

In [8]:
out

'hi this is a test<|endoftext|>this is another sentence'

In [9]:
text1

'hi this is a test<|endoftext|>this is another sentence'

In [14]:
text1 = "hi this is a test<|endoftext|>this is another sentence<|endoftext|>"

In [15]:
out = ""
for id in tok.encode(text1):
    out += bytes.decode(vocab[id], errors='replace')

out

'hi this is a test<|endoftext|>this is another sentence<|endoftext|>'

In [16]:
text1.split("<|endoftext|>")

['hi this is a test', 'this is another sentence', '']